# Approach B — Gradient Boosting (XGBoost) with Conformal Prediction

**Notebook 2 of 5** · Wireless Networks Project · Team Cupcake (Team 5)

---

## Objective

Train an **XGBoost regressor** to predict **Time-To-Congestion (TTC)** for WiFi Access Points, and wrap it with **Normalized Conformal Prediction** (via the `crepes` library) to produce **per-sample prediction intervals** with **guaranteed 90% marginal coverage**.

### Research Gaps Addressed
| Gap | Description | How |
|-----|-------------|-----|
| Gap 1 | No proactive/predictive load balancing | TTC regression enables proactive decisions |
| Gap 2 | No estimation of *time until* congestion | Explicit TTC label from Notebook 0 |
| Gap 3 | No confidence-aware decision making | Conformal Prediction intervals with coverage guarantees |

### Key Contrast with Approach A (TCN + MC Dropout)
- **Interpretable** tree-based model (vs. neural network)
- **Statistically rigorous** prediction intervals via Conformal Prediction (vs. approximate MC Dropout intervals)
- **Faster** training and inference
- **SHAP** feature importance for full model transparency

---

## 0. Environment Setup

In [ ]:
%%capture
!pip install optuna shap crepes pyarrow

In [ ]:
import os
import json
import warnings
import time
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

import xgboost as xgb
import optuna
import shap
from crepes import ConformalRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Plotting defaults ──
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})
sns.set_style('whitegrid')

# ── Paths ──
INPUT_DIR  = Path('/kaggle/input')       # where the dataset is mounted
WORK_DIR   = Path('/kaggle/working')
PROC_DIR   = WORK_DIR / 'processed'
SAVE_DIR   = WORK_DIR / 'model_b'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── GPU check ──
# XGBoost >= 2.0: use tree_method='hist' with device='cuda' (gpu_hist is deprecated)
gpu_available = os.path.exists('/dev/nvidia0') or os.popen('nvidia-smi').read().strip() != ''
DEVICE = 'cuda' if gpu_available else 'cpu'
print(f'XGBoost device: {DEVICE}')
if gpu_available:
    print('GPU detected — XGBoost will use hist tree method on CUDA device')
else:
    print('No GPU — falling back to CPU (hist tree method)')
print(f'XGBoost version: {xgb.__version__}')

---
## 1. Data Loading

Load the processed parquet files produced by **Notebook 0** (`00_data_preparation.ipynb`).  
Expected columns per row: `timestamp, ap_id, clients_connected, clients_delta, rssi_mean, rssi_std, channel_utilization, neighbor_avg_load, hour_of_day, day_of_week, ttc, source`.

In [ ]:
# ── Locate processed data ──
# Notebook 0 saves to /kaggle/working/processed/, but when added as a
# dataset to this notebook it appears under /kaggle/input/<dataset-slug>/processed/
# We search both locations.

def find_processed_dir():
    """Search for the processed/ directory in common Kaggle paths."""
    candidates = [
        PROC_DIR,                                        # same kernel run
        *INPUT_DIR.glob('*/processed'),                  # added as dataset
        *INPUT_DIR.glob('*/working/processed'),          # nested
    ]
    for p in candidates:
        if p.exists() and (p / 'train.parquet').exists():
            return p
    raise FileNotFoundError(
        'Could not find processed/ directory with train.parquet. '
        'Run Notebook 0 first or attach its output as a dataset.'
    )

proc_dir = find_processed_dir()
print(f'Using processed data from: {proc_dir}')

train_df = pd.read_parquet(proc_dir / 'train.parquet')
val_df   = pd.read_parquet(proc_dir / 'val.parquet')
test_df  = pd.read_parquet(proc_dir / 'test.parquet')

print(f'Train: {train_df.shape}  |  Val: {val_df.shape}  |  Test: {test_df.shape}')
train_df.head()

In [ ]:
# Quick sanity checks
print('Columns:', list(train_df.columns))
print(f'\nTTC stats (train):')
print(train_df['ttc'].describe())
print(f'\nNull TTC rows (train): {train_df["ttc"].isna().sum()}')
print(f'Source distribution (train):\n{train_df["source"].value_counts()}')

---
## 2. Feature Engineering — Lag Feature Flattening

XGBoost is a **tabular** model — it does not natively consume sequences.  
We flatten each AP's time window into a **lag-feature vector**: for each feature `f`, we create `f_lag_0, f_lag_1, ..., f_lag_9` (covering the same 10-step lookback window used by the TCN in Notebook 1).

This converts the temporal dimension into additional columns.

In [ ]:
# ── Configuration ──
SEQ_LEN = 10  # number of lag steps (must match Notebook 1 seq_len)

RAW_FEATURES = [
    'clients_connected', 'clients_delta',
    'channel_utilization', 'neighbor_avg_load',
    'hour_of_day', 'day_of_week',
]

TARGET = 'ttc'
META_COLS = ['timestamp', 'ap_id', 'source']  # kept for analysis, not used as features


def create_lag_features(df: pd.DataFrame, features: list, n_lags: int) -> pd.DataFrame:
    """
    For each AP, create lag columns (f_lag_0 = current, f_lag_1 = t-1, ...).
    Rows with insufficient history (first n_lags-1 per AP) are dropped.
    """
    df = df.sort_values(['ap_id', 'timestamp']).reset_index(drop=True)
    
    lag_frames = []
    for ap_id, grp in df.groupby('ap_id'):
        grp = grp.reset_index(drop=True)
        lag_dict = {}
        for feat in features:
            for lag in range(n_lags):
                lag_dict[f'{feat}_lag_{lag}'] = grp[feat].shift(lag)
        lag_df = pd.DataFrame(lag_dict)
        lag_df[TARGET] = grp[TARGET].values
        for mc in META_COLS:
            if mc in grp.columns:
                lag_df[mc] = grp[mc].values
        lag_frames.append(lag_df)
    
    result = pd.concat(lag_frames, ignore_index=True)
    
    # Drop rows where lags are NaN (first n_lags-1 per AP) and where TTC is NaN
    result = result.dropna().reset_index(drop=True)
    return result


print('Creating lag features (this may take a moment)...')
t0 = time.time()
train_lag = create_lag_features(train_df, RAW_FEATURES, SEQ_LEN)
val_lag   = create_lag_features(val_df,   RAW_FEATURES, SEQ_LEN)
test_lag  = create_lag_features(test_df,  RAW_FEATURES, SEQ_LEN)
elapsed = time.time() - t0

FEATURE_COLS = [c for c in train_lag.columns if c not in [TARGET] + META_COLS]

print(f'Done in {elapsed:.1f}s')
print(f'Train: {train_lag.shape}  |  Val: {val_lag.shape}  |  Test: {test_lag.shape}')
print(f'Number of features: {len(FEATURE_COLS)}')
print(f'Feature names (first 20): {FEATURE_COLS[:20]}')

---
## 3. Dedicated Calibration Set Construction

**Critical for valid conformal prediction:** The calibration set must be **separate from both the training set and the validation set** used for early stopping. Using the validation set for both early stopping *and* conformal calibration violates the exchangeability assumption because the model has already seen the val labels indirectly.

We carve out the **last 30% of the training set** (by time) as a dedicated calibration set. This preserves the time-ordered split from Notebook 0 and ensures:
- The model is trained on `train_proper` (70% of original train)
- Early stopping uses `val` (unchanged)
- Conformal calibration uses `cal` (carved from train, never seen during training)
- Test remains untouched

In [ ]:
# ── Split training data into train_proper + calibration ──
# IMPORTANT: create_lag_features() concatenates APs in groupby order, not
# global time order. We must sort by timestamp globally before the iloc
# split to ensure a true time-based partition (not an AP-based split).

CAL_FRACTION = 0.30  # fraction of training data reserved for conformal calibration

# Sort by timestamp globally so the split is truly time-based
train_lag = train_lag.sort_values('timestamp').reset_index(drop=True)

n_train_total = len(train_lag)
n_cal = int(n_train_total * CAL_FRACTION)
n_train_proper = n_train_total - n_cal

# Time-based: take the last portion as calibration (later time steps)
train_proper_lag = train_lag.iloc[:n_train_proper].reset_index(drop=True)
cal_lag          = train_lag.iloc[n_train_proper:].reset_index(drop=True)

print(f'Time range — train_proper: {train_proper_lag["timestamp"].min()} → {train_proper_lag["timestamp"].max()}')
print(f'Time range — calibration:  {cal_lag["timestamp"].min()} → {cal_lag["timestamp"].max()}')

# Prepare arrays
X_train = train_proper_lag[FEATURE_COLS].values
y_train = train_proper_lag[TARGET].values

X_cal = cal_lag[FEATURE_COLS].values
y_cal = cal_lag[TARGET].values

X_val = val_lag[FEATURE_COLS].values
y_val = val_lag[TARGET].values

X_test = test_lag[FEATURE_COLS].values
y_test = test_lag[TARGET].values

print(f'\nTrain (proper): {X_train.shape}  — used for model training')
print(f'Calibration:    {X_cal.shape}  — used ONLY for conformal calibration (never seen during training)')
print(f'Validation:     {X_val.shape}  — used for early stopping')
print(f'Test:           {X_test.shape}  — held out for final evaluation')
print(f'Total train rows: {n_train_total} → proper: {n_train_proper} + cal: {n_cal}')

---
## 4. Hyperparameter Optimization with Optuna

We run a **50-trial Optuna study** to find the best XGBoost hyperparameters, optimizing **validation MAE**.  
XGBoost ≥ 2.0 uses `tree_method='hist'` with `device='cuda'` (the old `gpu_hist` method is deprecated).

In [ ]:
N_OPTUNA_TRIALS = 50


def objective(trial):
    """Optuna objective: train XGBoost with sampled HPs, return val MAE."""
    params = {
        'objective': 'reg:pseudohubererror',  # Huber-like loss, robust to outliers
        'tree_method': 'hist',                # XGBoost >= 2.0: always 'hist'
        'device': DEVICE,                     # 'cuda' if GPU available, else 'cpu'
        'eval_metric': 'mae',
        'verbosity': 0,
        'seed': RANDOM_SEED,
        
        # ── Hyperparameters to tune ──
        'n_estimators':     trial.suggest_int('n_estimators', 100, 1500),
        'max_depth':        trial.suggest_int('max_depth', 3, 12),
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 50),
        'gamma':            trial.suggest_float('gamma', 1e-8, 5.0, log=True),
        'huber_slope':      trial.suggest_float('huber_slope', 0.5, 5.0),
    }
    
    model = xgb.XGBRegressor(**params, early_stopping_rounds=30)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )
    
    y_pred = model.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)
    return mae


print(f'Starting Optuna study with {N_OPTUNA_TRIALS} trials...')
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

print(f'\n✅ Best trial:')
print(f'   Val MAE = {study.best_value:.4f}')
print(f'   Params  = {json.dumps(study.best_params, indent=2)}')

In [ ]:
# ── Optuna visualizations ──
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1) Optimization history
trials = study.trials
trial_vals = [t.value for t in trials if t.value is not None]
best_so_far = np.minimum.accumulate(trial_vals)

axes[0].scatter(range(len(trial_vals)), trial_vals, alpha=0.4, s=20, label='Trial MAE')
axes[0].plot(best_so_far, color='crimson', linewidth=2, label='Best so far')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Validation MAE')
axes[0].set_title('Optuna Optimization History')
axes[0].legend()

# 2) Parameter importances (via fANOVA)
try:
    importances = optuna.importance.get_param_importances(study)
    params_sorted = list(importances.keys())
    values_sorted = list(importances.values())
    axes[1].barh(params_sorted[::-1], values_sorted[::-1], color='steelblue')
    axes[1].set_xlabel('Importance')
    axes[1].set_title('Hyperparameter Importance (fANOVA)')
except Exception:
    axes[1].text(0.5, 0.5, 'fANOVA not available', ha='center', va='center', fontsize=14)
    axes[1].set_title('Hyperparameter Importance')

plt.tight_layout()
plt.savefig(SAVE_DIR / 'optuna_history.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Train Final Model with Best Hyperparameters

Retrain using the best parameters found by Optuna, with early stopping on the validation set.

In [ ]:
best_params = study.best_params.copy()
best_params.update({
    'objective': 'reg:pseudohubererror',
    'tree_method': 'hist',     # XGBoost >= 2.0: always 'hist', never 'gpu_hist'
    'device': DEVICE,          # 'cuda' for GPU acceleration
    'eval_metric': 'mae',
    'verbosity': 1,
    'seed': RANDOM_SEED,
})

print('Training final XGBoost model with best hyperparameters...')
t0 = time.time()

model = xgb.XGBRegressor(**best_params, early_stopping_rounds=50)
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=100,
)

train_time = time.time() - t0
print(f'\n✅ Training complete in {train_time:.1f}s')
print(f'   Best iteration: {model.best_iteration}')
print(f'   Best val MAE:   {model.best_score:.4f}')

# Save model
model.save_model(str(SAVE_DIR / 'model_b_xgboost.json'))
print(f'   Model saved to: {SAVE_DIR / "model_b_xgboost.json"}')

---
## 6. Conformal Prediction — Normalized (Per-Sample) Intervals via `crepes`

### Why Normalized Conformal Prediction?

Standard (vanilla) split conformal produces a **constant** interval width for all test points: $[\hat{y} - \hat{q},\ \hat{y} + \hat{q}]$. This means every sample gets the same confidence score, making it useless for per-AP decision gating in the controller (Notebook 4).

**Normalized conformal prediction** scales intervals by a per-sample **difficulty estimate** $\sigma_i$:
- Nonconformity score: $s_i = \frac{|y_i - \hat{y}_i|}{\sigma_i}$
- Test interval: $[\hat{y} - \hat{q} \cdot \sigma,\ \hat{y} + \hat{q} \cdot \sigma]$

This produces **wider intervals for harder samples** and **narrower intervals for easier ones**, enabling meaningful per-sample confidence scores.

### Implementation
1. Compute **out-of-fold (OOF) residuals** via 5-fold CV on the training set to get realistic generalization errors (in-sample residuals are near-zero due to XGBoost memorization)
2. Train a **difficulty estimator** (secondary XGBoost) on OOF `|residual|` values
3. Use `crepes.ConformalRegressor` with the `sigmas` parameter for normalized calibration
4. At test time, per-sample $\sigma$ values yield per-sample interval widths

In [ ]:
# ── Step 1: Compute out-of-fold (OOF) residuals via 5-fold CV ──
# Why OOF? XGBoost memorizes training data to near-zero residual, so
# in-sample |residual| is artificially small. The difficulty estimator
# would learn "everything is easy" and produce underestimated sigmas,
# making intervals too narrow and breaking the coverage guarantee.
# OOF residuals reflect genuine generalization error.

N_OOF_FOLDS = 5
oof_residuals = np.zeros(len(X_train))
kf = KFold(n_splits=N_OOF_FOLDS, shuffle=False)  # no shuffle: preserve time ordering

print(f'Computing OOF residuals via {N_OOF_FOLDS}-fold CV...')
for fold_idx, (tr_idx, oof_idx) in enumerate(kf.split(X_train)):
    fold_model = xgb.XGBRegressor(
        **{k: v for k, v in best_params.items()},
        early_stopping_rounds=30,
    )
    fold_model.fit(
        X_train[tr_idx], y_train[tr_idx],
        eval_set=[(X_train[oof_idx], y_train[oof_idx])],
        verbose=False,
    )
    oof_preds = fold_model.predict(X_train[oof_idx])
    oof_residuals[oof_idx] = np.abs(y_train[oof_idx] - oof_preds)
    print(f'  Fold {fold_idx+1}/{N_OOF_FOLDS}: OOF MAE = {np.mean(oof_residuals[oof_idx]):.4f}')

print(f'\nOverall OOF MAE: {np.mean(oof_residuals):.4f}')
print(f'OOF residual range: [{oof_residuals.min():.4f}, {oof_residuals.max():.4f}]')

# ── Step 2: Train difficulty estimator on OOF residuals ──
print('\nTraining difficulty estimator (secondary XGBoost) on OOF residuals...')
difficulty_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    tree_method='hist',
    device=DEVICE,
    objective='reg:squarederror',
    seed=RANDOM_SEED + 1,
    verbosity=0,
)
difficulty_model.fit(X_train, oof_residuals)
print('Difficulty estimator trained on OOF |residuals|.')

# Predict difficulty (sigma) for calibration and test sets
sigmas_cal  = difficulty_model.predict(X_cal)
sigmas_test = difficulty_model.predict(X_test)

# Floor sigmas to avoid division by zero / extremely narrow intervals
SIGMA_FLOOR = 1e-4
sigmas_cal  = np.maximum(sigmas_cal,  SIGMA_FLOOR)
sigmas_test = np.maximum(sigmas_test, SIGMA_FLOOR)

print(f'\nCalibration sigmas: min={sigmas_cal.min():.4f}, median={np.median(sigmas_cal):.4f}, max={sigmas_cal.max():.4f}')
print(f'Test sigmas:        min={sigmas_test.min():.4f}, median={np.median(sigmas_test):.4f}, max={sigmas_test.max():.4f}')

In [ ]:
# ── Step 3: Conformal calibration on the DEDICATED calibration set ──
# This set was carved from training data and NEVER seen during model fitting.

y_cal_pred = model.predict(X_cal)
cal_residuals_abs = np.abs(y_cal - y_cal_pred)  # for reporting
cal_residuals_signed = y_cal - y_cal_pred        # crepes expects signed residuals

print(f'Calibration set size: {len(y_cal)}')
print(f'Calibration MAE:  {np.mean(cal_residuals_abs):.4f}')
print(f'Calibration RMSE: {np.sqrt(np.mean(cal_residuals_signed**2)):.4f}')

# ── Use crepes for normalized conformal prediction ──
# IMPORTANT: crepes expects SIGNED residuals (y_true - y_hat) as a positional arg.
# It computes absolute values internally for nonconformity scores.
cr = ConformalRegressor()
cr.fit(cal_residuals_signed, sigmas=sigmas_cal)
print('\n✅ crepes ConformalRegressor fitted with normalized (per-sample) difficulty estimates')

In [ ]:
# ── Visualize nonconformity score distribution ──
# Normalized scores = |residual| / sigma
normalized_scores = cal_residuals_abs / sigmas_cal

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1) Raw absolute residuals
axes[0].hist(cal_residuals_abs, bins=80, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('|y - ŷ| (raw residual)')
axes[0].set_ylabel('Count')
axes[0].set_title('Raw Calibration Residuals')

# 2) Difficulty estimates (sigmas)
axes[1].hist(sigmas_cal, bins=80, color='darkorange', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('σ (difficulty estimate)')
axes[1].set_ylabel('Count')
axes[1].set_title('Difficulty Estimates on Cal Set')

# 3) Normalized scores
axes[2].hist(normalized_scores, bins=80, color='seagreen', edgecolor='white', alpha=0.8)
# Mark the 90% and 80% quantiles
q90_norm = np.quantile(normalized_scores, np.ceil(0.9 * (len(normalized_scores) + 1)) / len(normalized_scores))
q80_norm = np.quantile(normalized_scores, np.ceil(0.8 * (len(normalized_scores) + 1)) / len(normalized_scores))
axes[2].axvline(q90_norm, color='crimson', linestyle='--', linewidth=2, label=f'90% q={q90_norm:.2f}')
axes[2].axvline(q80_norm, color='orange', linestyle='--', linewidth=2, label=f'80% q={q80_norm:.2f}')
axes[2].set_xlabel('Normalized Score |y - ŷ| / σ')
axes[2].set_ylabel('Count')
axes[2].set_title('Normalized Nonconformity Scores')
axes[2].legend()

plt.tight_layout()
plt.savefig(SAVE_DIR / 'conformal_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Test-Time Inference — Point Predictions + Per-Sample Conformal Intervals

In [ ]:
# ── Point predictions ──
t0 = time.time()
y_test_pred = model.predict(X_test)
inference_time = time.time() - t0

# ── Per-sample conformal intervals via crepes ──
# crepes.ConformalRegressor.predict returns an ndarray of shape (n, 2) = [lower, upper]
intervals_90 = cr.predict(y_hat=y_test_pred, sigmas=sigmas_test, confidence=0.90)
intervals_80 = cr.predict(y_hat=y_test_pred, sigmas=sigmas_test, confidence=0.80)

test_lower_90 = intervals_90[:, 0]
test_upper_90 = intervals_90[:, 1]
test_lower_80 = intervals_80[:, 0]
test_upper_80 = intervals_80[:, 1]

# ── Per-sample confidence score ──
# Interval width now VARIES per sample (thanks to normalized conformal).
# Confidence = 1 / (1 + interval_width_90 / max_ttc)
# Narrower interval → higher confidence → more suitable for handover gating.
interval_widths_90 = test_upper_90 - test_lower_90
max_ttc = max(y_train.max(), 1.0)
confidence_scores = 1.0 / (1.0 + interval_widths_90 / max_ttc)

print(f'Inference time: {inference_time*1000:.1f}ms for {len(X_test)} samples')
print(f'\n90% interval widths: min={interval_widths_90.min():.4f}, '
      f'median={np.median(interval_widths_90):.4f}, max={interval_widths_90.max():.4f}')
print(f'Confidence scores:   min={confidence_scores.min():.4f}, '
      f'median={np.median(confidence_scores):.4f}, max={confidence_scores.max():.4f}')
print(f'Confidence std:      {confidence_scores.std():.4f}  (>0 confirms per-sample variation)')

---
## 8. Evaluation Metrics

In [ ]:
# ── Core regression metrics ──
mae  = mean_absolute_error(y_test, y_test_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
r2   = r2_score(y_test, y_test_pred)

# ── Coverage metrics ──
coverage_90 = np.mean((y_test >= test_lower_90) & (y_test <= test_upper_90))
coverage_80 = np.mean((y_test >= test_lower_80) & (y_test <= test_upper_80))

# ── Interval width ──
mean_width_90 = np.mean(interval_widths_90)
mean_width_80 = np.mean(test_upper_80 - test_lower_80)

# ── Interval efficiency (coverage / width ratio) ──
efficiency_90 = coverage_90 / mean_width_90 if mean_width_90 > 0 else 0

print('=' * 60)
print('       APPROACH B — XGBoost + Conformal Prediction')
print('=' * 60)
print(f'  MAE              : {mae:.4f} intervals')
print(f'  RMSE             : {rmse:.4f} intervals')
print(f'  R²               : {r2:.4f}')
print('-' * 60)
print(f'  90% Coverage     : {coverage_90:.4f}  (target ≥ 0.90)')
print(f'  90% Mean Width   : {mean_width_90:.4f}  (varies per sample)')
print(f'  80% Coverage     : {coverage_80:.4f}  (target ≥ 0.80)')
print(f'  80% Mean Width   : {mean_width_80:.4f}  (varies per sample)')
print('-' * 60)
print(f'  Training time    : {train_time:.1f}s')
print(f'  Inference time   : {inference_time*1000:.1f}ms ({len(X_test)} samples)')
print(f'  Device           : {DEVICE}')
print(f'  Conformal method : Normalized split conformal (crepes)')
print('=' * 60)

---
## 9. Source-Stratified Analysis (Real vs. Synthetic)

Since Notebook 0 tags each row with a `source` flag (`real` or `synthetic`), we break down MAE and coverage separately to validate the hybrid data strategy. This is critical for Notebook 5 and directly addresses the examiner challenge: *"How do you know the synthetic data is realistic?"*

In [ ]:
# ── Source-stratified evaluation ──
source_metrics = {}

if 'source' in test_lag.columns:
    test_sources = test_lag['source'].values
    unique_sources = np.unique(test_sources)
    print(f'Sources found in test set: {unique_sources}')
    print(f'Distribution: {dict(zip(*np.unique(test_sources, return_counts=True)))}\n')
    
    for src in unique_sources:
        mask = test_sources == src
        n_src = mask.sum()
        
        y_true_src = y_test[mask]
        y_pred_src = y_test_pred[mask]
        lower_90_src = test_lower_90[mask]
        upper_90_src = test_upper_90[mask]
        lower_80_src = test_lower_80[mask]
        upper_80_src = test_upper_80[mask]
        widths_90_src = interval_widths_90[mask]
        conf_src = confidence_scores[mask]
        
        mae_src  = mean_absolute_error(y_true_src, y_pred_src)
        rmse_src = np.sqrt(mean_squared_error(y_true_src, y_pred_src))
        r2_src   = r2_score(y_true_src, y_pred_src) if n_src > 1 else float('nan')
        cov_90   = np.mean((y_true_src >= lower_90_src) & (y_true_src <= upper_90_src))
        cov_80   = np.mean((y_true_src >= lower_80_src) & (y_true_src <= upper_80_src))
        
        source_metrics[src] = {
            'n_samples': int(n_src),
            'mae': float(mae_src),
            'rmse': float(rmse_src),
            'r2': float(r2_src),
            'coverage_90': float(cov_90),
            'coverage_80': float(cov_80),
            'mean_interval_width_90': float(np.mean(widths_90_src)),
            'mean_confidence': float(np.mean(conf_src)),
        }
        
        print(f'── {src.upper()} ({n_src} samples) ──')
        print(f'  MAE:          {mae_src:.4f}')
        print(f'  RMSE:         {rmse_src:.4f}')
        print(f'  R²:           {r2_src:.4f}')
        print(f'  90% Coverage: {cov_90:.4f}')
        print(f'  80% Coverage: {cov_80:.4f}')
        print(f'  Mean Width:   {np.mean(widths_90_src):.4f}')
        print(f'  Mean Conf:    {np.mean(conf_src):.4f}')
        print()
else:
    print('Warning: No "source" column found in test data. Skipping stratified analysis.')
    test_sources = None

In [ ]:
# ── Source-stratified bar charts ──
if source_metrics:
    sources = list(source_metrics.keys())
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    x = np.arange(len(sources))
    bar_width = 0.5
    colors = {'real': 'steelblue', 'synthetic': 'darkorange'}
    bar_colors = [colors.get(s, 'gray') for s in sources]
    
    # MAE comparison
    maes = [source_metrics[s]['mae'] for s in sources]
    axes[0].bar(x, maes, bar_width, color=bar_colors, edgecolor='white')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([s.capitalize() for s in sources])
    axes[0].set_ylabel('MAE')
    axes[0].set_title('MAE by Data Source')
    for i, v in enumerate(maes):
        axes[0].text(i, v + max(maes)*0.02, f'{v:.3f}', ha='center', fontsize=11)
    
    # Coverage comparison
    covs = [source_metrics[s]['coverage_90'] for s in sources]
    axes[1].bar(x, covs, bar_width, color=bar_colors, edgecolor='white')
    axes[1].axhline(0.9, color='crimson', linestyle='--', linewidth=2, label='90% target')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([s.capitalize() for s in sources])
    axes[1].set_ylabel('Empirical Coverage')
    axes[1].set_title('90% Coverage by Data Source')
    axes[1].legend()
    for i, v in enumerate(covs):
        axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=11)
    
    # Interval width comparison
    widths = [source_metrics[s]['mean_interval_width_90'] for s in sources]
    axes[2].bar(x, widths, bar_width, color=bar_colors, edgecolor='white')
    axes[2].set_xticks(x)
    axes[2].set_xticklabels([s.capitalize() for s in sources])
    axes[2].set_ylabel('Mean Interval Width')
    axes[2].set_title('Mean 90% Interval Width by Source')
    for i, v in enumerate(widths):
        axes[2].text(i, v + max(widths)*0.02, f'{v:.3f}', ha='center', fontsize=11)
    
    plt.suptitle('Real vs. Synthetic Data: Model Performance Comparison', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(SAVE_DIR / 'source_stratified.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Skipping source visualization (no source column).')

In [ ]:
# ── Save metrics for Notebook 5 comparison ──
metrics = {
    'model': 'Approach B — XGBoost + Conformal Prediction',
    'mae': float(mae),
    'rmse': float(rmse),
    'r2': float(r2),
    'coverage_90': float(coverage_90),
    'coverage_80': float(coverage_80),
    'interval_width_90': float(mean_width_90),
    'interval_width_80': float(mean_width_80),
    'training_time_s': float(train_time),
    'inference_time_ms': float(inference_time * 1000),
    'device': DEVICE,
    'conformal_method': 'normalized_split_conformal_crepes',
    'best_params': study.best_params,
    'n_optuna_trials': N_OPTUNA_TRIALS,
    'n_features': len(FEATURE_COLS),
    'n_train': len(X_train),
    'n_cal': len(X_cal),
    'n_val': len(X_val),
    'n_test': len(X_test),
    'source_stratified': source_metrics,  # per-source MAE, coverage, etc.
}

with open(SAVE_DIR / 'metrics_b.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'Metrics saved to {SAVE_DIR / "metrics_b.json"}')

---
## 10. Visualizations

In [ ]:
# ── 10a. Predicted vs. True TTC (scatter) ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot
sample_idx = np.random.choice(len(y_test), size=min(5000, len(y_test)), replace=False)
axes[0].scatter(y_test[sample_idx], y_test_pred[sample_idx], s=8, alpha=0.3, color='steelblue')
lims = [min(y_test.min(), y_test_pred.min()), max(y_test.max(), y_test_pred.max())]
axes[0].plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('True TTC (intervals)')
axes[0].set_ylabel('Predicted TTC (intervals)')
axes[0].set_title(f'Predicted vs True TTC\nMAE={mae:.3f}, R²={r2:.3f}')
axes[0].legend()

# Residual distribution
residuals = y_test - y_test_pred
axes[1].hist(residuals, bins=80, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='crimson', linestyle='--', linewidth=2)
axes[1].set_xlabel('Residual (True - Predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Residual Distribution\nMean={np.mean(residuals):.3f}, Std={np.std(residuals):.3f}')

plt.tight_layout()
plt.savefig(SAVE_DIR / 'pred_vs_true.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 10b. TTC Predictions with Per-Sample Conformal Intervals (time series view) ──

# Select a single AP for visualization
if 'ap_id' in test_lag.columns:
    sample_ap = test_lag['ap_id'].value_counts().idxmax()  # AP with most test points
    ap_mask = (test_lag['ap_id'] == sample_ap).values
else:
    # fallback: use first N points
    ap_mask = np.zeros(len(test_lag), dtype=bool)
    ap_mask[:min(200, len(test_lag))] = True

# Limit to 200 points for readability
ap_indices = np.where(ap_mask)[0][:200]

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

x_axis = np.arange(len(ap_indices))
true_vals = y_test[ap_indices]
pred_vals = y_test_pred[ap_indices]
lower_90 = test_lower_90[ap_indices]
upper_90 = test_upper_90[ap_indices]
lower_80 = test_lower_80[ap_indices]
upper_80 = test_upper_80[ap_indices]
conf_vals = confidence_scores[ap_indices]

# Top: TTC with intervals
ax = axes[0]
ax.fill_between(x_axis, lower_90, upper_90, alpha=0.15, color='steelblue', label=f'90% CI (coverage={coverage_90:.2%})')
ax.fill_between(x_axis, lower_80, upper_80, alpha=0.25, color='steelblue', label=f'80% CI (coverage={coverage_80:.2%})')
ax.plot(x_axis, true_vals, 'o-', markersize=3, color='crimson', linewidth=1, label='True TTC', alpha=0.8)
ax.plot(x_axis, pred_vals, 's-', markersize=3, color='navy', linewidth=1, label='Predicted TTC', alpha=0.8)
ax.set_ylabel('TTC (intervals)')
ap_label = f' (AP {sample_ap})' if 'ap_id' in test_lag.columns else ''
ax.set_title(f'TTC Prediction with Per-Sample Conformal Intervals{ap_label}')
ax.legend(loc='upper right')

# Bottom: Per-sample confidence scores
ax2 = axes[1]
ax2.fill_between(x_axis, conf_vals, alpha=0.3, color='seagreen')
ax2.plot(x_axis, conf_vals, color='seagreen', linewidth=1.5)
ax2.axhline(0.7, color='crimson', linestyle='--', linewidth=1.5, label='Handover gate threshold (0.7)')
ax2.set_xlabel('Time Step')
ax2.set_ylabel('Confidence Score')
ax2.set_title('Per-Sample Confidence Scores (for Controller Gating)')
ax2.set_ylim(0, 1.05)
ax2.legend()

plt.tight_layout()
plt.savefig(SAVE_DIR / 'conformal_intervals_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 10c. Coverage vs. Interval Width Trade-off Curve ──
# Sweep confidence levels and plot empirical coverage vs mean interval width

conf_levels = np.arange(0.50, 0.996, 0.01)
coverages = []
widths = []

for cl in conf_levels:
    intervals_sweep = cr.predict(y_hat=y_test_pred, sigmas=sigmas_test, confidence=cl)
    lower_sweep = intervals_sweep[:, 0]
    upper_sweep = intervals_sweep[:, 1]
    cov = np.mean((y_test >= lower_sweep) & (y_test <= upper_sweep))
    wid = np.mean(upper_sweep - lower_sweep)
    coverages.append(cov)
    widths.append(wid)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Coverage vs target
axes[0].plot(conf_levels, coverages, 'o-', markersize=3, color='steelblue', linewidth=2)
axes[0].plot([0.5, 1.0], [0.5, 1.0], 'r--', linewidth=1.5, label='Ideal (coverage = target)')
axes[0].axhline(0.9, color='orange', linestyle=':', alpha=0.7, label='90% target')
axes[0].set_xlabel('Target Coverage Level')
axes[0].set_ylabel('Empirical Coverage')
axes[0].set_title('Conformal Coverage Calibration')
axes[0].legend()

# Width vs coverage
axes[1].plot(coverages, widths, 'o-', markersize=3, color='steelblue', linewidth=2)
axes[1].axvline(0.9, color='orange', linestyle=':', alpha=0.7, label='90% coverage')
axes[1].set_xlabel('Empirical Coverage')
axes[1].set_ylabel('Mean Interval Width')
axes[1].set_title('Interval Width vs Coverage Trade-off')
axes[1].legend()

plt.tight_layout()
plt.savefig(SAVE_DIR / 'coverage_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 10d. Confidence Score Distribution ──
# Demonstrate that confidence scores genuinely vary per sample
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Distribution
axes[0].hist(confidence_scores, bins=60, color='seagreen', edgecolor='white', alpha=0.8)
axes[0].axvline(0.7, color='crimson', linestyle='--', linewidth=2, label='Handover gate threshold')
axes[0].axvline(confidence_scores.mean(), color='navy', linestyle=':', linewidth=2, label=f'Mean={confidence_scores.mean():.3f}')
axes[0].set_xlabel('Confidence Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Per-Sample Confidence Scores')
axes[0].legend()

# Confidence vs interval width
axes[1].scatter(interval_widths_90[sample_idx], confidence_scores[sample_idx],
                s=8, alpha=0.3, color='steelblue')
axes[1].set_xlabel('90% Interval Width')
axes[1].set_ylabel('Confidence Score')
axes[1].set_title('Confidence vs Interval Width (inverse relationship)')

plt.tight_layout()
plt.savefig(SAVE_DIR / 'confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary stats
n_high_conf = (confidence_scores > 0.7).sum()
n_low_conf  = (confidence_scores <= 0.7).sum()
print(f'High-confidence samples (>0.7): {n_high_conf} ({n_high_conf/len(confidence_scores):.1%})')
print(f'Low-confidence samples (≤0.7):  {n_low_conf} ({n_low_conf/len(confidence_scores):.1%})')

---
## 11. Feature Importance — SHAP Analysis

SHAP (SHapley Additive exPlanations) values provide a principled, game-theoretic decomposition of each prediction into feature contributions. This is a key advantage of XGBoost over neural network approaches.

In [ ]:
# ── Compute SHAP values ──
# Use a subsample for efficiency (SHAP on full test set can be slow)
SHAP_SAMPLE_SIZE = min(2000, len(X_test))
shap_idx = np.random.choice(len(X_test), size=SHAP_SAMPLE_SIZE, replace=False)
X_shap = X_test[shap_idx]

print(f'Computing SHAP values for {SHAP_SAMPLE_SIZE} samples...')
t0 = time.time()
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_shap)
print(f'Done in {time.time() - t0:.1f}s')

In [ ]:
# ── 11a. SHAP Beeswarm Plot (Top 20 features) ──
plt.figure(figsize=(12, 10))
shap.summary_plot(
    shap_values, X_shap,
    feature_names=FEATURE_COLS,
    max_display=20,
    show=False,
    plot_size=(12, 10),
)
plt.title('SHAP Feature Importance — Beeswarm Plot (Top 20)', fontsize=14, pad=20)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 11b. SHAP Bar Plot (mean |SHAP|) ──
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values, X_shap,
    feature_names=FEATURE_COLS,
    plot_type='bar',
    max_display=20,
    show=False,
    plot_size=(10, 8),
)
plt.title('Mean |SHAP| — Feature Importance Ranking (Top 20)', fontsize=14, pad=20)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 11c. SHAP Dependence Plots for Top 4 Features ──
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top4_idx = np.argsort(mean_abs_shap)[-4:][::-1]
top4_names = [FEATURE_COLS[i] for i in top4_idx]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, feat_idx, feat_name in zip(axes.flat, top4_idx, top4_names):
    shap.dependence_plot(
        feat_idx, shap_values, X_shap,
        feature_names=FEATURE_COLS,
        ax=ax,
        show=False,
    )
    ax.set_title(f'SHAP Dependence: {feat_name}', fontsize=12)

plt.suptitle('SHAP Dependence Plots — Top 4 Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 12. Grouped Analysis — Feature Importance by Category

Group SHAP importances by the original feature (across all lag steps) to understand which *types* of information matter most.

In [ ]:
# ── Group SHAP values by original feature name ──
grouped_importance = {}
for i, col in enumerate(FEATURE_COLS):
    # Extract base feature name (e.g., 'clients_connected' from 'clients_connected_lag_3')
    base = '_'.join(col.split('_')[:-2]) if '_lag_' in col else col
    if base not in grouped_importance:
        grouped_importance[base] = 0.0
    grouped_importance[base] += mean_abs_shap[i]

# Sort and plot
sorted_groups = sorted(grouped_importance.items(), key=lambda x: x[1], reverse=True)
group_names = [g[0] for g in sorted_groups]
group_values = [g[1] for g in sorted_groups]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(group_names[::-1], group_values[::-1], color='steelblue', edgecolor='white')
ax.set_xlabel('Summed Mean |SHAP| Across All Lags')
ax.set_title('Feature Group Importance (Aggregated Across Lag Steps)')

# Add value labels
for bar, val in zip(bars, group_values[::-1]):
    ax.text(bar.get_width() + max(group_values) * 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'grouped_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nFeature group importance (descending):')
for name, val in sorted_groups:
    print(f'  {name:25s} : {val:.4f}')

---
## 13. Lag Importance Analysis

Which time steps (lags) contribute the most to the prediction? This reveals the model's effective "memory" horizon.

In [ ]:
# ── Importance by lag position ──
lag_importance = np.zeros(SEQ_LEN)
for i, col in enumerate(FEATURE_COLS):
    if '_lag_' in col:
        lag_num = int(col.split('_lag_')[-1])
        lag_importance[lag_num] += mean_abs_shap[i]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(SEQ_LEN), lag_importance, color='steelblue', edgecolor='white')
bars[0].set_color('crimson')  # highlight current time step

ax.set_xlabel('Lag Step (0 = current, 9 = oldest)')
ax.set_ylabel('Summed Mean |SHAP|')
ax.set_title('Importance by Lag Position — How Far Back Does the Model Look?')
ax.set_xticks(range(SEQ_LEN))
ax.set_xticklabels([f't' if i == 0 else f't-{i}' for i in range(SEQ_LEN)])

plt.tight_layout()
plt.savefig(SAVE_DIR / 'lag_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 14. Error Analysis by TTC Range

How does model performance vary across different TTC ranges? This is crucial for understanding whether the model is reliable for urgent (low TTC) vs. comfortable (high TTC) predictions.

In [ ]:
# ── Error analysis by TTC bin ──
ttc_bins = pd.qcut(y_test, q=5, duplicates='drop')
error_analysis = pd.DataFrame({
    'y_true': y_test,
    'y_pred': y_test_pred,
    'residual': residuals,
    'abs_error': np.abs(residuals),
    'in_90_interval': (y_test >= test_lower_90) & (y_test <= test_upper_90),
    'ttc_bin': ttc_bins,
})

bin_metrics = error_analysis.groupby('ttc_bin', observed=True).agg(
    count=('y_true', 'count'),
    mae=('abs_error', 'mean'),
    rmse=('residual', lambda x: np.sqrt(np.mean(x**2))),
    mean_bias=('residual', 'mean'),
    coverage_90=('in_90_interval', 'mean'),
).round(4)

print('Performance by TTC Range:')
print(bin_metrics.to_string())

# ── Visualization ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

bin_labels = [str(b) for b in bin_metrics.index]

axes[0].bar(bin_labels, bin_metrics['mae'], color='steelblue', edgecolor='white')
axes[0].set_xlabel('TTC Range')
axes[0].set_ylabel('MAE')
axes[0].set_title('MAE by TTC Range')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(bin_labels, bin_metrics['coverage_90'], color='steelblue', edgecolor='white')
axes[1].axhline(0.9, color='crimson', linestyle='--', linewidth=2, label='90% target')
axes[1].set_xlabel('TTC Range')
axes[1].set_ylabel('Empirical Coverage')
axes[1].set_title('90% Coverage by TTC Range')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=30)

axes[2].bar(bin_labels, bin_metrics['mean_bias'], color='steelblue', edgecolor='white')
axes[2].axhline(0, color='crimson', linestyle='--', linewidth=1.5)
axes[2].set_xlabel('TTC Range')
axes[2].set_ylabel('Mean Bias (True - Pred)')
axes[2].set_title('Prediction Bias by TTC Range')
axes[2].tick_params(axis='x', rotation=30)

plt.suptitle('Error Analysis by TTC Range', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'error_by_ttc_range.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 15. Comparison of Interval Quality vs. Approach A

This section provides a standalone comparison between Approach B (XGBoost + Conformal Prediction) and Approach A (TCN + MC Dropout). If Notebook 1 has been run, we load its metrics for a direct comparison. Otherwise, we provide a **theoretical comparison** based on the known properties of each uncertainty method — ensuring this notebook remains independently runnable.

In [ ]:
# ── Load Approach A metrics if available ──
model_a_metrics_path = WORK_DIR / 'model_a' / 'metrics_a.json'
# Also check input directory (if Notebook 1 output is attached as dataset)
alt_paths = list(INPUT_DIR.glob('*/model_a/metrics_a.json')) + \
            list(INPUT_DIR.glob('*/working/model_a/metrics_a.json'))

model_a_metrics = None
for path in [model_a_metrics_path] + alt_paths:
    if path.exists():
        with open(path) as f:
            model_a_metrics = json.load(f)
        print(f'✅ Loaded Approach A metrics from: {path}')
        break

if model_a_metrics is None:
    print('ℹ️  Approach A metrics not found — using theoretical comparison.')
    print('   (Run Notebook 1 first or attach its output as a dataset for empirical comparison.)')

In [ ]:
# ── Build comparison table ──
comparison_rows = [
    ['MAE',               f'{mae:.4f}'],
    ['RMSE',              f'{rmse:.4f}'],
    ['R²',                f'{r2:.4f}'],
    ['90% Coverage',      f'{coverage_90:.4f}'],
    ['90% Mean Width',    f'{mean_width_90:.4f}'],
    ['80% Coverage',      f'{coverage_80:.4f}'],
    ['80% Mean Width',    f'{mean_width_80:.4f}'],
    ['Training Time',     f'{train_time:.1f}s'],
    ['Uncertainty Method', 'Normalized Conformal (crepes)'],
    ['Per-Sample Intervals', 'Yes (via difficulty estimation)'],
    ['Coverage Guarantee', 'Yes (finite-sample marginal)'],
    ['Interpretability',  'High (SHAP)'],
]

if model_a_metrics:
    # Empirical comparison
    a_col = [
        f"{model_a_metrics.get('mae', 'N/A')}",
        f"{model_a_metrics.get('rmse', 'N/A')}",
        f"{model_a_metrics.get('r2', 'N/A')}",
        f"{model_a_metrics.get('coverage_90', 'N/A')}",
        f"{model_a_metrics.get('interval_width_90', 'N/A')}",
        f"{model_a_metrics.get('coverage_80', 'N/A')}",
        f"{model_a_metrics.get('interval_width_80', 'N/A')}",
        f"{model_a_metrics.get('training_time_s', 'N/A')}s",
        'MC Dropout (T=50)',
        'Yes (via prediction std)',
        'No (approximate)',
        'Low-Medium',
    ]
    comparison_df = pd.DataFrame(
        [[r[0], a, r[1]] for r, a in zip(comparison_rows, a_col)],
        columns=['Metric', 'Approach A (TCN+MCD)', 'Approach B (XGB+CP)'],
    )
else:
    # Theoretical comparison (standalone mode)
    a_col_theoretical = [
        '(run NB1)',    # MAE
        '(run NB1)',    # RMSE
        '(run NB1)',    # R²
        '(run NB1)',    # 90% Coverage
        '(run NB1)',    # 90% Width
        '(run NB1)',    # 80% Coverage
        '(run NB1)',    # 80% Width
        '(run NB1)',    # Training time
        'MC Dropout (T=50)',
        'Yes (via prediction std)',
        'No (approximate)',
        'Low-Medium',
    ]
    comparison_df = pd.DataFrame(
        [[r[0], a, r[1]] for r, a in zip(comparison_rows, a_col_theoretical)],
        columns=['Metric', 'Approach A (TCN+MCD)', 'Approach B (XGB+CP)'],
    )

print('\n' + '=' * 80)
print('  APPROACH A vs B — INTERVAL QUALITY COMPARISON')
print('=' * 80)
print(comparison_df.to_string(index=False))

# Save for Notebook 5
comparison_df.to_csv(SAVE_DIR / 'comparison_a_vs_b.csv', index=False)
print(f'\nSaved to {SAVE_DIR / "comparison_a_vs_b.csv"}')

In [ ]:
# ── Theoretical comparison discussion (always printed, regardless of NB1 availability) ──
print('╔' + '═' * 72 + '╗')
print('║  INTERVAL QUALITY: CONFORMAL PREDICTION vs MC DROPOUT              ║')
print('╠' + '═' * 72 + '╣')
print('║                                                                        ║')
print('║  Conformal Prediction (Approach B):                                    ║')
print('║  ✓ Guaranteed marginal coverage: P(Y ∈ CI) ≥ 1-α (finite-sample)      ║')
print('║  ✓ No distributional assumptions required                              ║')
print('║  ✓ With normalization: per-sample interval widths                      ║')
print('║  ✗ Requires a held-out calibration set (reduces training data)         ║')
print('║  ✗ Coverage guarantee is marginal (not conditional on X)               ║')
print('║                                                                        ║')
print('║  MC Dropout (Approach A):                                              ║')
print('║  ✓ Naturally per-sample (std from T forward passes)                    ║')
print('║  ✓ No calibration set needed                                           ║')
print('║  ✓ Flexible: works with any dropout-enabled neural network             ║')
print('║  ✗ No formal coverage guarantee                                        ║')
print('║  ✗ Computationally expensive (T=50 forward passes per prediction)      ║')
print('║  ✗ Quality depends on dropout placement and rate                       ║')
print('║                                                                        ║')
print('║  For safety-critical handover decisions, conformal prediction\'s         ║')
print('║  coverage guarantee makes it the more trustworthy choice.              ║')
print('╚' + '═' * 72 + '╝')

---
## 16. Export Predictions for Notebook 4 (Controller Simulation)

Save test-set predictions, intervals, and confidence scores so Notebook 4 can use the best model for the proactive controller simulation.

In [ ]:
# ── Save predictions for downstream use ──
preds_df = pd.DataFrame({
    'y_true': y_test,
    'y_pred': y_test_pred,
    'lower_90': test_lower_90,
    'upper_90': test_upper_90,
    'lower_80': test_lower_80,
    'upper_80': test_upper_80,
    'confidence': confidence_scores,
    'interval_width_90': interval_widths_90,
    'sigma': sigmas_test,
})

# Add metadata if available
if 'ap_id' in test_lag.columns:
    preds_df['ap_id'] = test_lag['ap_id'].values
if 'timestamp' in test_lag.columns:
    preds_df['timestamp'] = test_lag['timestamp'].values
if 'source' in test_lag.columns:
    preds_df['source'] = test_lag['source'].values

preds_df.to_parquet(SAVE_DIR / 'predictions_b.parquet', index=False)
print(f'Predictions saved: {SAVE_DIR / "predictions_b.parquet"}  ({len(preds_df)} rows)')

# ── Save the fitted crepes ConformalRegressor via joblib ──
# This allows Notebook 4 to reconstruct per-sample intervals for new AP
# states without re-running calibration (cr is not JSON-serializable).
joblib.dump(cr, SAVE_DIR / 'conformal_regressor.joblib')
print(f'Conformal regressor saved: {SAVE_DIR / "conformal_regressor.joblib"}')
print('  → Notebook 4 can reload with: cr = joblib.load("model_b/conformal_regressor.joblib")')

# Save conformal config metadata
conformal_config = {
    'method': 'normalized_split_conformal',
    'library': 'crepes',
    'n_calibration_samples': int(len(y_cal)),
    'calibration_mae': float(np.mean(cal_residuals_abs)),
    'sigma_floor': float(SIGMA_FLOOR),
    'cal_fraction': float(CAL_FRACTION),
    'n_oof_folds': N_OOF_FOLDS,
    'usage_in_nb4': 'cr = joblib.load(model_b/conformal_regressor.joblib); intervals = cr.predict(y_hat, sigmas=sigmas, confidence=0.9)',
}
with open(SAVE_DIR / 'conformal_config.json', 'w') as f:
    json.dump(conformal_config, f, indent=2)
print(f'Conformal config saved: {SAVE_DIR / "conformal_config.json"}')

# Save difficulty estimator
difficulty_model.save_model(str(SAVE_DIR / 'difficulty_estimator.json'))
print(f'Difficulty estimator saved: {SAVE_DIR / "difficulty_estimator.json"}')

---
## 17. Summary

### Key Results

In [ ]:
print('╔' + '═' * 62 + '╗')
print('║  NOTEBOOK 2 — APPROACH B: XGBoost + Conformal Prediction     ║')
print('╠' + '═' * 62 + '╣')
print(f'║  MAE:                {mae:<40.4f} ║')
print(f'║  RMSE:               {rmse:<40.4f} ║')
print(f'║  R²:                 {r2:<40.4f} ║')
print('╠' + '═' * 62 + '╣')
print(f'║  90% Coverage:       {coverage_90:<40.4f} ║')
print(f'║  90% Mean Width:     {mean_width_90:<40.4f} ║')
print(f'║  80% Coverage:       {coverage_80:<40.4f} ║')
print(f'║  80% Mean Width:     {mean_width_80:<40.4f} ║')
print('╠' + '═' * 62 + '╣')
print(f'║  Training Time:      {train_time:<40.1f} ║')
print(f'║  Optuna Trials:      {N_OPTUNA_TRIALS:<40d} ║')
print(f'║  Device:             {DEVICE:<40s} ║')
print(f'║  Features:           {len(FEATURE_COLS):<40d} ║')
print(f'║  Conformal Method:   {"Normalized (crepes)":<40s} ║')
print(f'║  Calibration Set:    {"Dedicated (separate from val)":<40s} ║')
print(f'║  Per-Sample Scores:  {"Yes (OOF + difficulty estimation)":<40s} ║')
print('╚' + '═' * 62 + '╝')
print()
print('Key insights:')
print('  1. Normalized conformal prediction produces PER-SAMPLE interval widths,')
print('     enabling meaningful confidence gating in the proactive controller.')
print('  2. Conformal intervals have GUARANTEED marginal coverage (≥90%),')
print('     unlike MC Dropout which provides only approximate uncertainty.')
print('  3. A dedicated calibration set (separate from val) ensures the')
print('     coverage guarantee is not invalidated by data snooping.')
print('  4. OOF residuals for difficulty estimation prevent the model from')
print('     learning artificially small sigmas from memorized training data.')
print('  5. SHAP analysis provides full transparency into which features')
print('     and lag steps drive each prediction.')

### Saved Artifacts

| File | Description |
|------|-------------|
| `model_b/model_b_xgboost.json` | Trained XGBoost model (best Optuna params) |
| `model_b/difficulty_estimator.json` | Difficulty estimator for normalized conformal |
| `model_b/conformal_regressor.joblib` | Fitted crepes ConformalRegressor (for Notebook 4) |
| `model_b/metrics_b.json` | All evaluation metrics incl. source-stratified (for Notebook 5) |
| `model_b/predictions_b.parquet` | Test predictions + per-sample intervals + confidence |
| `model_b/conformal_config.json` | Conformal method config + NB4 usage instructions |
| `model_b/comparison_a_vs_b.csv` | Side-by-side with Approach A |
| `model_b/*.png` | All visualizations |

### Key Takeaways

1. **XGBoost + Normalized Conformal Prediction** provides a fully interpretable, fast-training approach to TTC regression with uncertainty quantification.
2. **Normalized conformal** (via `crepes` + OOF difficulty estimation) produces **per-sample** interval widths that vary by input difficulty — essential for per-AP handover gating in Notebook 4.
3. **Out-of-fold residuals** ensure the difficulty estimator sees genuine generalization errors, not memorized training fit — preventing underestimated sigmas and broken coverage.
4. **SHAP analysis** reveals which features and lag steps drive predictions — something not easily obtained from the TCN approach.
5. **Conformal intervals** offer a mathematically grounded coverage guarantee, making them preferable for safety-critical handover decisions.
6. The **dedicated calibration set** (carved from training data, sorted by global timestamp, separate from val/test) ensures the coverage guarantee is not compromised.
7. **Source-stratified metrics** (real vs. synthetic) are logged in `metrics_b.json` for direct use by Notebook 5.
8. The fitted `ConformalRegressor` is **serialized via joblib**, allowing Notebook 4 to reconstruct intervals without re-running calibration.

---
*Proceed to Notebook 3 (`03_model_gnnlite_xgb_ensemble.ipynb`) for the spatial extension.*